In [1]:
import os
import sys
import zipfile

# Add the project root to the system path so Python can find 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import DataIngestionConfig
from src.audio.entity.artifact_entity import DataIngestionArtifacts
from src.audio.cloud_storage.s3_operations import S3Sync

In [8]:
class DataIngestion:
    def __init__(self, data_ingestion_config: DataIngestionConfig):
        try:
            self.data_ingestion_config = data_ingestion_config
            self.s3_sync = S3Sync()
            self.data_ingestion_artifact = self.data_ingestion_config.data_ingestion_artifact_dir
        except Exception as e:
            raise CustomException(e, sys)

    def get_data_from_cloud(self):
        try:
            logging.info("Initiating data download from s3 bucket")
            print(("Initiating data download from s3 bucket"))
            download_dir = self.data_ingestion_config.download_dir
            zip_file_path = self.data_ingestion_config.zip_data_path

            if os.path.exists(zip_file_path):
                logging.info(f"Data already exists at {zip_file_path}. Skipping download")
                print(f"Data already exists at {zip_file_path}. Skipping download")
            else:
                os.makedirs(download_dir, exist_ok=True)
                logging.info(f"Data downloading from {S3_BUCKET_DATA_URI} to {download_dir}")
                print(f"Data downloading from {S3_BUCKET_DATA_URI} to {download_dir}")
                self.s3_sync.sync_folder_from_s3(folder=download_dir, aws_bucket_url=S3_BUCKET_DATA_URI)
                logging.info("data downloaded successfully from s3 bucket")
        except Exception as e:
            logging.error(f"Error occurred while downloading data from s3 bucket: {e}")
            raise CustomException(e, sys)

    def unzip_data(self):
        try:
            logging.info('Unzipping downloaded file')
            print('Unzipping downloaded file')
            raw_zip_path = self.data_ingestion_config.zip_data_path
            unzip_dir = self.data_ingestion_config.unzip_data_dir
            data_dir = os.path.join(unzip_dir, "data")
            if os.path.isdir(data_dir):
                logging.info(f"unzipped folder already exists at {data_dir}")
                print(f"unzipped folder already exists at {data_dir}")
            else:
                os.makedirs(data_dir, exist_ok=True)
                with zipfile.ZipFile(raw_zip_path, "r") as f:
                    f.extractall(data_dir)
                logging.info(f"Unzipping complete. Extracted to {data_dir}")
        except Exception as e:
            logging.error("Failed to unzip")
            raise CustomException(e, sys)

    def initiate_data_ingestion(self):
        try:
            logging.info("Initiating Data Ingestion")
            os.makedirs(self.data_ingestion_artifact, exist_ok=True)
            self.get_data_from_cloud()
            self.unzip_data()
            data_ingestion_artifact = DataIngestionArtifacts(data_folder_path=self.data_ingestion_config.unzip_data_dir)
            logging.info("Data ingestion complete")
            return data_ingestion_artifact
        except Exception as e:
            logging.error("Ingestion failed")
            raise CustomException(e, sys)

In [3]:
ingest = DataIngestion(DataIngestionConfig)

In [5]:
print("Expected folder:", DataIngestionConfig.download_dir)
print("Expected zip path:", DataIngestionConfig.zip_data_path)

Expected folder: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\data
Expected zip path: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\data\data.zip


In [6]:
import os
import wave
import struct
import math

# Target path expected by unzip_data()
base_data_dir = os.path.join(DataIngestionConfig.unzip_data_dir, "data")

# Create two class folders ('0' and '1') with 5 sample .wav files each
for label, freq in [("0", 440.0), ("1", 880.0)]:
    class_dir = os.path.join(base_data_dir, label)
    os.makedirs(class_dir, exist_ok=True)
    
    for i in range(5):
        file_path = os.path.join(class_dir, f"sample_{i}.wav")
        with wave.open(file_path, "w") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(16000)
            # Generate 1 second of audio tone
            frames = [
                struct.pack("<h", int(32767.0 * math.sin(2.0 * math.pi * freq * t / 16000)))
                for t in range(16000)
            ]
            wav_file.writeframes(b"".join(frames))

print("Sample dataset created successfully at:", base_data_dir)

Sample dataset created successfully at: C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data


In [7]:
ingest.unzip_data()

Unzipping downloaded file
unzipped folder already exists at C:\Users\User\OneDrive\Documents\Desktop\PW Data Science\Audio Binary Classification-20251029T060518Z-1-001\Audio-Classification-main\Audio-Classification-main\artifacts\data_ingestion\unzip\data
